In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
sensors = pd.read_csv("../data/mdm2_data_files/big_table_with_weather_rain_clusters_cctv300m.csv")

# keep unique sensors only
sensors = sensors[["sensor_id","latitude","longitude"]].drop_duplicates()

sensors.head()

,sensor_id,latitude,longitude
0,1,51.453815,-2.591538
8248,2,51.453865,-2.592004
16496,4,51.453281,-2.595743
24745,5,51.453285,-2.590382
32992,6,51.453117,-2.590430


In [3]:
gdf_sensors = gpd.GeoDataFrame(
    sensors,
    geometry=gpd.points_from_xy(sensors.longitude, sensors.latitude),
    crs="EPSG:4326"
)

In [4]:
wards = gpd.read_file("../data/wards/Wards.geojson")

wards.head()

,OBJECTID,NAME,COUNCILLORS,WARD_ID,SHAPESTArea,SHAPESTLength,geometry
0,1,Avonmouth & Lawrence Weston,3,E05010886,2.143475e+07,24446.971267,"POLYGON ((-2.72171 51.50167, -2.72156 51.50166..."
1,2,Stoke Bishop,2,E05010916,5.731849e+06,11049.516431,"POLYGON ((-2.65887 51.48407, -2.65807 51.48337..."
2,3,Henbury & Brentry,2,E05010901,3.889154e+06,12729.361309,"POLYGON ((-2.64717 51.51217, -2.64725 51.51201..."
3,4,Bishopsworth,2,E05010889,3.881891e+06,9386.242820,"POLYGON ((-2.63393 51.42749, -2.6339 51.42729,..."
4,5,Hartcliffe & Withywood,3,E05010900,3.609795e+06,10446.140970,"POLYGON ((-2.62874 51.416, -2.6288 51.41588, -..."


In [5]:
sensor_wards = gpd.sjoin(
    gdf_sensors,
    wards,
    how="left",
    predicate="within"
)

sensor_wards.head()

,sensor_id,latitude,longitude,geometry,index_right,OBJECTID,NAME,COUNCILLORS,WARD_ID,SHAPESTArea,SHAPESTLength
0,1,51.453815,-2.591538,POINT (-2.59154 51.45382),29,30,Central,2,E05010892,2.257391e+06,8258.086671
8248,2,51.453865,-2.592004,POINT (-2.592 51.45386),29,30,Central,2,E05010892,2.257391e+06,8258.086671
16496,4,51.453281,-2.595743,POINT (-2.59574 51.45328),29,30,Central,2,E05010892,2.257391e+06,8258.086671
24745,5,51.453285,-2.590382,POINT (-2.59038 51.45328),29,30,Central,2,E05010892,2.257391e+06,8258.086671
32992,6,51.453117,-2.590430,POINT (-2.59043 51.45312),29,30,Central,2,E05010892,2.257391e+06,8258.086671


In [6]:
survey = pd.read_csv("../data/surveys/Bristol_Quality_of_Life_data_2024_25.csv")

survey.head()

,Year,Indicator_Reference_Number,Priority_Indicator,Public_Title,Theme,Polarity,Bristol_average,Sample_Size,Standard_Error,Lower_Confidence_Limit,...,ICS_code,Ward_Group_Statistic,Standard_Error_1,Lower_Confidence_Limit_2,Upper_Confidence_Limit_3,Sample_Size_response,p_value,Statistical_Significance,Gap,FID
0,2023,IQOL17189,False,% who feel Bristol City Council does not provi...,Council & Democracy,↓ Lower value is better,47.4,3789.0,0.934193,45.5,...,NaN,46.7,4.801147,37.5,56.1,132.0,NaN,No difference,0.7,1
1,2023,IQOL17190,True,% satisfied with the way Bristol City Council ...,Council & Democracy,↑ Higher value is better,34.1,3860.0,0.888635,32.4,...,NaN,33.3,4.460591,25.2,42.5,134.0,NaN,No difference,0.8,2
2,2023,IQOL17191,False,% dissatisfied with the way Bristol City Counc...,Council & Democracy,↓ Lower value is better,40.7,3860.0,0.900528,38.9,...,NaN,39.3,4.548324,30.8,48.5,134.0,NaN,No difference,1.4,3
3,2023,IQOL17205,False,% in full time paid work,Economy,No benefit in higher or lower value,65.7,2426.0,1.020530,63.7,...,NaN,72.7,4.351457,63.4,80.4,95.0,NaN,No difference,7.0,4
4,2023,IQOL17206,False,% in part time paid work,Economy,No benefit in higher or lower value,23.3,2426.0,0.945637,21.5,...,NaN,17.1,3.803779,10.8,25.8,95.0,NaN,No difference,6.2,5


In [8]:
# load unique sensors from your big table
big = pd.read_csv("../data/mdm2_data_files/big_table_with_weather_rain_clusters_cctv300m.csv")
sensors = big[["sensor_id", "longitude", "latitude"]].drop_duplicates("sensor_id").copy()

gdf_sensors = gpd.GeoDataFrame(
    sensors,
    geometry=gpd.points_from_xy(sensors["longitude"], sensors["latitude"]),
    crs="EPSG:4326"
)

wards = gpd.read_file("../data/wards/Wards.geojson")  # ward name is wards["NAME"]

sensor_wards = gpd.sjoin(gdf_sensors, wards[["NAME", "geometry"]], how="left", predicate="within")
print(sensor_wards[["sensor_id", "NAME"]].head())
print("Sensors with ward assigned:", sensor_wards["NAME"].notna().mean())

       sensor_id     NAME
0              1  Central
8248           2  Central
16496          4  Central
24745          5  Central
32992          6  Central
Sensors with ward assigned: 1.0


In [10]:
qol = pd.read_csv("../data/surveys/Bristol_Quality_of_Life_data_2024_25.csv")

# the safety perception question we want
SAFETY_Q = "% who feel safe in their local area after dark"

# filter to the correct year and question
qol_safety = qol[
    (qol["Year"] == 2024) &
    (qol["Public_Title"] == SAFETY_Q) &
    (qol["Group_"].isna()) &
    (qol["Ward_name"].notna())
].copy()

# keep only relevant columns
qol_safety = qol_safety[["Ward_name", "Ward_Group_Statistic"]].rename(
    columns={"Ward_Group_Statistic": "safety_after_dark_pct"}
)

print(qol_safety.head())
print("Number of wards in safety data:", qol_safety["Ward_name"].nunique())

                         Ward_name  safety_after_dark_pct
76991                       Ashley                   49.3
76992  Avonmouth & Lawrence Weston                   60.3
76993                   Bedminster                   70.1
76994     Bishopston & Ashley Down                   74.6
76995                 Bishopsworth                   47.0
Number of wards in safety data: 34


In [11]:
# normalize names to avoid small spelling differences
sensor_wards["ward_key"] = sensor_wards["NAME"].str.strip().str.lower()
qol_safety["ward_key"] = qol_safety["Ward_name"].str.strip().str.lower()

# merge
sensor_safety = sensor_wards.merge(
    qol_safety[["ward_key", "safety_after_dark_pct"]],
    on="ward_key",
    how="left"
)

sensor_safety_out = sensor_safety[["sensor_id", "NAME", "safety_after_dark_pct"]].rename(
    columns={"NAME": "ward_name"}
)

print(sensor_safety_out.head())
print("Sensors with safety value:", sensor_safety_out["safety_after_dark_pct"].notna().mean())

   sensor_id ward_name  safety_after_dark_pct
0          1   Central                   53.5
1          2   Central                   53.5
2          4   Central                   53.5
3          5   Central                   53.5
4          6   Central                   53.5
Sensors with safety value: 1.0


In [12]:
OUT = "../reports/sensor_ward_safety_after_dark_2024.csv"

sensor_safety_out.to_csv(OUT, index=False)

print("Saved:", OUT)

Saved: ../reports/sensor_ward_safety_after_dark_2024.csv


In [13]:
import pandas as pd

df = pd.read_csv("../data/mdm2_data_files/big_table_with_weather_rain_clusters_cctv300m.csv")

print("Unique sensors in dataset:", df["sensor_id"].nunique())

print("\nSensors per cluster:")
print(df.groupby("cluster_geo")["sensor_id"].nunique())

Unique sensors in dataset: 46

Sensors per cluster:
cluster_geo
0    24
1     6
2    16
Name: sensor_id, dtype: int64


In [14]:
safety = pd.read_csv("../reports/sensor_ward_safety_after_dark_2024.csv")

print("Sensors in safety file:", safety["sensor_id"].nunique())
print("Rows:", len(safety))

Sensors in safety file: 46
Rows: 46


In [15]:
# load the modelling dataset (contains the sensors actually used)
big = pd.read_csv("../data/mdm2_data_files/big_table_with_weather_rain_clusters_cctv300m.csv")

valid_sensors = big["sensor_id"].unique()

# keep only sensors used in the model
sensor_safety_filtered = sensor_safety_out[
    sensor_safety_out["sensor_id"].isin(valid_sensors)
]

print("Sensors kept:", sensor_safety_filtered["sensor_id"].nunique())

# save final dataset
OUT = "../reports/sensor_ward_safety_after_dark_2024.csv"
sensor_safety_filtered.to_csv(OUT, index=False)

print("Saved:", OUT)

Sensors kept: 46
Saved: ../reports/sensor_ward_safety_after_dark_2024.csv
